# Bellabeat Smart Device Data Analysis
### Case Study: Marketing Recommendations for New User Acquisition

This notebook conducts descriptive analysis and feature engineering on Fitbit activity, sleep, and weight tracking data combined from March to May 2016. The insights will help shape marketing strategies for Bellabeat's line of health-focused smart products.

## Phase 4: Analyze
In this section, we load the processed combined data, calculate baseline statistics, classify users based on health guidelines, analyze hourly/daily activity cycles, and export the final engineered tables.

In [1]:
import pandas as pd
import numpy as np

# Load processed datasets
df_daily = pd.read_csv('processed_data/daily_merged.csv')
df_hourly = pd.read_csv('processed_data/hourly_merged.csv')

print(f"Daily merged loaded: {df_daily.shape} rows")
print(f"Hourly merged loaded: {df_hourly.shape} rows")

Daily merged loaded: (1373, 23) rows
Hourly merged loaded: (46008, 6) rows


### 1. General Descriptive Statistics
Let's look at average daily steps, active time, sleep, and caloric burn to establish baseline metrics.

In [2]:
summary_cols = ['total_steps', 'calories', 'very_active_minutes', 'fairly_active_minutes', 
                'lightly_active_minutes', 'sedentary_minutes', 'total_minutes_asleep']
print("=== Descriptive Summary Statistics ===")
print(df_daily[summary_cols].describe().round(2))

=== Descriptive Summary Statistics ===
       total_steps  calories  ...  sedentary_minutes  total_minutes_asleep
count      1373.00   1373.00  ...            1373.00                590.00
mean       7247.36   2264.45  ...             993.43                419.59
std        5214.78    754.16  ...             314.79                120.54
min           0.00      0.00  ...               0.00                 56.00
25%        3108.00   1793.00  ...             729.00                363.25
50%        6910.00   2114.00  ...            1058.00                431.50
75%       10538.00   2766.00  ...            1247.00                490.00
max       36019.00   4900.00  ...            1440.00                976.00

[8 rows x 7 columns]


### 2. Feature Engineering & Tier Classifications
We classify users according to recognized standards:
- **Activity Tiers (WHO step levels)**: Sedentary (<5k), Lightly Active (5k-7.5k), Fairly Active (7.5k-10k), Active/Highly Active (>=10k steps/day).
- **Sleep Tiers (NSF Guidelines)**: Insufficient (<7h), Sufficient (7-9h), Oversleeping (>9h).
- **Wear Consistency Tiers**: High Wear Consistency (worn >=80% days), Moderate (50-80% days), Low (<50% days).

In [3]:
# 1. Extract day and weekend metrics
df_daily['date'] = pd.to_datetime(df_daily['date'])
df_daily['day_name'] = df_daily['date'].dt.day_name()
df_daily['is_weekend'] = df_daily['date'].dt.dayofweek.isin([5, 6]).astype(int)

# 2. Activity Level Classification
def classify_activity(steps):
    if steps < 5000:
        return 'Sedentary (<5k steps)'
    elif steps < 7500:
        return 'Lightly Active (5k-7.5k steps)'
    elif steps < 10000:
        return 'Fairly Active (7.5k-10k steps)'
    else:
        return 'Active / Highly Active (>=10k steps)'

df_daily['activity_tier'] = df_daily['total_steps'].apply(classify_activity)

# 3. Sleep Sufficiency Classification
def classify_sleep(minutes):
    if pd.isna(minutes):
        return np.nan
    elif minutes < 420:
        return 'Insufficient (<7h)'
    elif minutes <= 540:
        return 'Sufficient (7-9h)'
    else:
        return 'Oversleep (>9h)'

df_daily['sleep_tier'] = df_daily['total_minutes_asleep'].apply(classify_sleep)

# 4. Wear Consistency
total_study_days = (df_daily['date'].max() - df_daily['date'].min()).days + 1
user_wear_days = df_daily.groupby('id')['date'].nunique().reset_index()
user_wear_days = user_wear_days.rename(columns={'date': 'days_worn'})
user_wear_days['wear_rate'] = user_wear_days['days_worn'] / total_study_days

def classify_consistency(rate):
    if rate >= 0.8:
        return 'High Wear Consistency (>=80% days)'
    elif rate >= 0.5:
        return 'Moderate Wear Consistency (50%-80% days)'
    else:
        return 'Low Wear Consistency (<50% days)'

user_wear_days['wear_tier'] = user_wear_days['wear_rate'].apply(classify_consistency)

# Merge consistency metrics
df_daily = pd.merge(df_daily, user_wear_days[['id', 'days_worn', 'wear_rate', 'wear_tier']], on='id', how='left')

print("Feature Engineering completed.")
print("\nWear Consistency Breakdown:")
print(user_wear_days['wear_tier'].value_counts())
print("\nDaily Activity Tiers:")
print(df_daily['activity_tier'].value_counts())

Feature Engineering completed.

Wear Consistency Breakdown:
wear_tier
Moderate Wear Consistency (50%-80% days)    30
Low Wear Consistency (<50% days)             4
High Wear Consistency (>=80% days)           1
Name: count, dtype: int64

Daily Activity Tiers:
activity_tier
Sedentary (<5k steps)                   496
Active / Highly Active (>=10k steps)    420
Lightly Active (5k-7.5k steps)          250
Fairly Active (7.5k-10k steps)          207
Name: count, dtype: int64


### 3. Weekday vs. Weekend Behaviors
Do users show different physical activity and sleeping behaviors on the weekends compared to weekdays?

In [4]:
print("=== Weekday vs. Weekend Averages ===")
grouped_weekend = df_daily.groupby('is_weekend')[['total_steps', 'calories', 'very_active_minutes', 'total_minutes_asleep', 'total_time_in_bed']].mean().round(2)
grouped_weekend.index = ['Weekday', 'Weekend']
print(grouped_weekend)

=== Weekday vs. Weekend Averages ===
         total_steps  calories  ...  total_minutes_asleep  total_time_in_bed
Weekday      7270.97   2259.29  ...                411.53             447.60
Weekend      7188.27   2277.36  ...                439.33             486.76

[2 rows x 5 columns]


### 4. Correlation Analysis
We investigate whether daily sedentary minutes, steps, or vigorous activity have a linear correlation with sleep duration.

In [5]:
correlation_steps_sleep = df_daily['total_steps'].corr(df_daily['total_minutes_asleep'])
correlation_sedentary_sleep = df_daily['sedentary_minutes'].corr(df_daily['total_minutes_asleep'])

print(f"Correlation between Daily Steps and Sleep: {correlation_steps_sleep:.4f}")
print(f"Correlation between Sedentary Minutes and Sleep: {correlation_sedentary_sleep:.4f}")

Correlation between Daily Steps and Sleep: -0.2089
Correlation between Sedentary Minutes and Sleep: -0.5259


### 5. Hourly Activity Profiling
We analyze hourly aggregates to understand peak active times during the day. This is highly actionable for marketing push notifications.

In [6]:
df_hourly['activity_hour'] = pd.to_datetime(df_hourly['activity_hour'])
df_hourly['hour'] = df_hourly['activity_hour'].dt.hour

hourly_profile = df_hourly.groupby('hour')[['step_total', 'calories', 'total_intensity']].mean().round(2)
print("=== Top 5 Most Active Hours (by Step Total) ===")
print(hourly_profile.sort_values(by='step_total', ascending=False).head(5))

=== Top 5 Most Active Hours (by Step Total) ===
      step_total  calories  total_intensity
hour                                       
19        554.89    118.79            20.27
18        550.27    119.14            20.14
12        534.26    115.32            19.17
14        506.02    113.50            17.91
17        499.71    117.28            19.64


### 6. Export Engineered Data for Streamlit App
Finally, we save these newly engineered dataframes so that the Streamlit app doesn't need to recompute these features at runtime, accelerating load time.

In [7]:
# Convert datetime columns back to ISO format string
df_daily['date'] = df_daily['date'].dt.strftime('%Y-%m-%d')
df_hourly['activity_hour'] = df_hourly['activity_hour'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Export datasets
df_daily.to_csv('processed_data/daily_engineered.csv', index=False)
df_hourly.to_csv('processed_data/hourly_engineered.csv', index=False)
print("Engineered datasets successfully written to processed_data/")

Engineered datasets successfully written to processed_data/
